# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inshrah-malik/inshrah-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Load dataset

In [4]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 21.6MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 8.93MB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [5]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature Vector

The feature vector contains only information that is available before predicting CTR. I selected traffic, search visibility, engagement, and user behavior features. Missing values are filled with 0 because unavailable metrics indicate no recorded activity instead of future information.

No future information or target-derived fields are included.

In [6]:
import pandas as pd

df = ds["train"].select(range(1000)).to_pandas()

feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "sessions_organic",
    "sessions_direct",
    "sessions_social",
    "scroll_events"
]

X = df[feature_cols].copy()
X = X.fillna(0)

print("Shape:", X.shape)
print(X.head())

Shape: (1000, 9)
   gsc_impressions  gsc_avg_position  ga4_pageviews  ga4_sessions  ga4_users  \
0               30          3.833333              0             0          0   
1                5         71.600000              0             0          0   
2                1         34.000000              0             0          0   
3                6         23.333333              0             0          0   
4                5         17.800000              0             0          0   

   sessions_organic  sessions_direct  sessions_social  scroll_events  
0                 0                0                0              0  
1                 0                0                0              0  
2                 0                0                0              0  
3                 0                0                0              0  
4                 0                0                0              0  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature          | Meaning                             | Missing Handling | Available Before Prediction? |
| ---------------- | ----------------------------------- | ---------------- | ---------------------------- |
| gsc_impressions  | Number of Google Search impressions | Fill with 0      | Yes                          |
| gsc_avg_position | Average search ranking position     | Fill with 0      | Yes                          |
| ga4_pageviews    | Total page views                    | Fill with 0      | Yes                          |
| ga4_sessions     | Total sessions                      | Fill with 0      | Yes                          |
| ga4_users        | Number of users                     | Fill with 0      | Yes                          |
| sessions_organic | Organic search sessions             | Fill with 0      | Yes                          |
| sessions_direct  | Direct traffic sessions             | Fill with 0      | Yes                          |
| sessions_social  | Social media sessions               | Fill with 0      | Yes                          |
| scroll_events    | User scroll events                  | Fill with 0      | Yes                          |

All selected features describe past observed website performance and can exist before predicting CTR.

In [8]:
print(X.info())

print("\nMissing values")

print(X.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gsc_impressions   1000 non-null   int64  
 1   gsc_avg_position  1000 non-null   float64
 2   ga4_pageviews     1000 non-null   int64  
 3   ga4_sessions      1000 non-null   int64  
 4   ga4_users         1000 non-null   int64  
 5   sessions_organic  1000 non-null   int64  
 6   sessions_direct   1000 non-null   int64  
 7   sessions_social   1000 non-null   int64  
 8   scroll_events     1000 non-null   int64  
dtypes: float64(1), int64(8)
memory usage: 70.4 KB
None

Missing values
gsc_impressions     0
gsc_avg_position    0
ga4_pageviews       0
ga4_sessions        0
ga4_users           0
sessions_organic    0
sessions_direct     0
sessions_social     0
scroll_events       0
dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage Check

I checked every selected feature for possible leakage.

Potential leakage can happen if a feature:

directly contains the target value
contains future information
is calculated after prediction time
contains manual labels

None of my selected features directly reveal CTR or use future information.

In [10]:
print("Columns in dataset")

for col in df.columns:
    print(col)

Columns in dataset
report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events


In [11]:
# Look for suspicious column names

possible_leakage = []

keywords = [
    "ctr",
    "target",
    "label",
    "future",
    "click_rate",
    "prediction"
]

for col in df.columns:
    lower = col.lower()
    if any(k in lower for k in keywords):
        possible_leakage.append(col)

print("Possible leakage columns:")
print(possible_leakage)

Possible leakage columns:
[]


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Field              | Reason                                                 |
| ------------------ | ------------------------------------------------------ |
| report_date        | Identifier only, not predictive by itself              |
| client_hash_id     | Identifier; would not generalize                       |
| content_hash_id    | Identifier; would cause memorization                   |
| client_has_gsc     | Metadata flag, not page performance                    |
| client_has_ga4     | Metadata flag                                          |
| gsc_data_available | Availability indicator, not behavioral signal          |
| ga4_data_available | Availability indicator                                 |
| sessions_paid      | Paid traffic is outside my CTR prediction focus        |
| sessions_ai        | AI referral traffic not directly related to search CTR |
| ai_chatgpt         | Traffic source only                                    |
| ai_perplexity      | Traffic source only                                    |
| ai_gemini          | Traffic source only                                    |
| ai_copilot         | Traffic source only                                    |
| ai_claude          | Traffic source only                                    |
| ai_meta            | Traffic source only                                    |
| ai_other           | Traffic source only                                    |

These fields were excluded because they are identifiers, metadata, or traffic-source flags that are less useful for predicting CTR in this baseline model.

In [13]:
excluded = [c for c in df.columns if c not in feature_cols]

print("Excluded columns")

for c in excluded:
    print("-", c)

Excluded columns
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_clicks
- gsc_sum_position
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_referral
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.